<div style="width: 100%; text-align: center;">
    <div style="background-color:#007F00; padding: 0.5rem;">
        <h1 style="font-weight: bold; font-size: 2.5em; color: black;"> AGRHYMET CENTRE CLIMATIQUE REGIONAL POUR L'AFRIQUE DE L'OUEST ET LE SAHEL</h1>
   </div>

   <div style="text-align: center;">
  <img src="https://www.sareco.org/wp-content/uploads/2017/07/plrDvYX1.jpg" width="200">
</div>

<a id="1"></a>
### <p style="padding:10px;background-color:#000000 ;margin:0;color:#007F00;font-family:#newtimeroman;font-size:100%;text-align:center;border-radius: 15px 50px;overflow:hidden;font-weight:500"> Modelisation hydrologique GR4J (demonstration atelier) </p>

## Calage et validation d'un modele pluie-debit GR4J

Ce notebook reprend, cellule par cellule, le script autonome
`gr4j/gr4j_modelling.R` : il permet de suivre chaque etape du calage et de
la validation d'un modele **GR4J** (package `airGR`) sans avoir a executer
tout le script d'un bloc, et d'inspecter les resultats intermediaires
(donnees chargees, parametres cales, indicateurs de performance) au fur et
a mesure.

Par rapport a un usage "recherche" (validation croisee a plusieurs blocs,
optimiseur externe DEoptim, sauvegarde en base SQLite), ce notebook est
volontairement simplifie pour la demonstration : une seule periode de
calage et une seule periode de validation independante, calibration
interne `Calibration_Michel` (rapide), export CSV/PNG.

**Donnees d'entree attendues** : un CSV avec les colonnes `date` (AAAA-MM-JJ),
`pcp` (precipitation, mm/j), `evap` (ETP, mm/j) et `debit` (debit observe a
l'exutoire, m3/s). Le pipeline de preparation de ces entrees a partir de
GloFAS (debit de reference) et d'une reanalyse (pluie/ETP) sera developpe
separement ; en attendant, ce notebook utilise `donnees_exemple_gr4j.csv`
(genere par `generer_donnees_exemple.py`), un jeu de donnees **entierement
fictif** — pas une vraie serie du bassin du Mouhoun — qui permet de tester
des maintenant tout l'enchainement.

**Important** : ce notebook necessite un noyau Jupyter **R** (`IRkernel`),
different du noyau Python utilise par `download_glofas_data.ipynb`. Voir
la section « Prerequis » du README (`scripts/utils/README.md`, § 6) pour
l'installation.

### 0. Packages requis

`airGR` (modele GR4J), `dplyr`/`tidyr`/`lubridate`/`readr` (manipulation des
donnees), `ggplot2` (graphiques), `hydroGOF` (indicateurs de performance
NSE/KGE/PBIAS/RMSE).

In [ ]:
packages_requis <- c("airGR", "dplyr", "tidyr", "lubridate", "readr", "ggplot2", "hydroGOF")
packages_manquants <- packages_requis[!sapply(packages_requis, requireNamespace, quietly = TRUE)]
if (length(packages_manquants) > 0) {
  install.packages(packages_manquants)
}
invisible(lapply(packages_requis, library, character.only = TRUE))

cat("Packages charges :", paste(packages_requis, collapse = ", "), "\n")


### 1. Parametres utilisateur

A adapter a votre bassin et vos donnees : identification, superficie (pour
convertir le debit observe de m3/s en mm/j), decoupage temporel
(mise en route / calage / validation) et critere de calage. Les valeurs par
defaut ci-dessous correspondent au jeu de donnees exemple.

In [ ]:
# --- Identification (utilise pour les titres et les noms de fichiers) ---
nom_bassin     <- "Bassin_Test"                  # ex. "Mouhoun" une fois vos vraies donnees branchees
fichier_entree <- "donnees_exemple_gr4j.csv"      # CSV : date, pcp, evap, debit
dossier_sortie <- "resultats_gr4j"                # dossier de sortie (cree automatiquement)

# --- Caracteristique du bassin versant ---
superficie_km2 <- 5000                            # superficie (km2)

# --- Decoupage temporel (adapter a la periode couverte par votre fichier) ---
date_debut_mise_en_route <- "2000-01-01"
date_debut_calage        <- "2001-01-01"
date_fin_calage          <- "2007-12-31"
date_debut_validation    <- "2008-01-01"
date_fin_validation      <- "2009-12-31"

# --- Critere de calage : un parmi "NSE", "KGE", "KGE2012" ---
critere_calage <- "KGE2012"


### 2. Chargement et controle des donnees

Verifications indispensables avant de lancer airGR : colonnes attendues,
continuite de la serie journaliere (pas de trou de date), absence de
valeurs manquantes dans la pluie et l'ETP. Le debit observe est ensuite
converti de m3/s vers mm/j.

In [ ]:
if (!file.exists(fichier_entree)) {
  stop(
    "Fichier d'entree introuvable : ", fichier_entree,
    "\nVerifiez le chemin, ou generez le jeu de donnees d'exemple avec generer_donnees_exemple.py"
  )
}

donnees <- read_csv(fichier_entree, show_col_types = FALSE)

colonnes_requises <- c("date", "pcp", "evap", "debit")
colonnes_absentes <- setdiff(colonnes_requises, names(donnees))
if (length(colonnes_absentes) > 0) {
  stop(
    "Colonne(s) manquante(s) dans ", fichier_entree, " : ", paste(colonnes_absentes, collapse = ", "),
    "\nColonnes attendues : ", paste(colonnes_requises, collapse = ", ")
  )
}

donnees <- donnees %>%
  mutate(date = as.Date(date)) %>%
  arrange(date) %>%
  distinct(date, .keep_all = TRUE)

# --- Continuite temporelle : airGR exige une serie journaliere sans trou ---
ecarts_dates <- diff(donnees$date)
if (any(ecarts_dates != 1)) {
  trous <- donnees$date[which(ecarts_dates != 1) + 1]
  stop(
    length(trous), " rupture(s) dans la serie temporelle (dates non consecutives), ",
    "par exemple autour de : ", paste(head(trous, 3), collapse = ", "),
    ".\nComblez les lacunes avant de poursuivre."
  )
}

# --- Valeurs manquantes sur les forcages (non tolerees par airGR) ---
n_na_pcp  <- sum(is.na(donnees$pcp))
n_na_evap <- sum(is.na(donnees$evap))
if (n_na_pcp > 0 || n_na_evap > 0) {
  stop(
    "Valeurs manquantes dans les forcages : ", n_na_pcp, " en pcp, ", n_na_evap, " en evap.\n",
    "Comblez-les avant de poursuivre (airGR n'accepte pas de NA dans P/ETP)."
  )
}

# --- Conversion du debit observe de m3/s vers mm/j ---
superficie_m2     <- superficie_km2 * 10^6
coef_m3s_vers_mmj <- (86400 * 1000) / superficie_m2
donnees <- donnees %>%
  mutate(debit_mm = debit * coef_m3s_vers_mmj)

cat(
  "Periode disponible :", format(min(donnees$date)), "->", format(max(donnees$date)),
  "(", nrow(donnees), "jours )\n"
)

head(donnees)


### 3. Preparation des entrees et des periodes pour airGR

airGR attend des dates au format `POSIXct`. On definit ensuite les indices
(positions dans `donnees`) correspondant a chaque periode : mise en route,
calage, validation.

In [ ]:
dates_posix <- as.POSIXct(donnees$date, tz = "UTC")

InputsModel <- CreateInputsModel(
  FUN_MOD = RunModel_GR4J,
  DatesR  = dates_posix,
  Precip  = donnees$pcp,
  PotEvap = donnees$evap
)

trouver_indices <- function(date_debut, date_fin, description) {
  indices <- which(donnees$date >= as.Date(date_debut) & donnees$date <= as.Date(date_fin))
  if (length(indices) == 0) {
    stop(
      "Aucune donnee entre ", date_debut, " et ", date_fin, " (periode '", description, "')",
      " -- verifiez que ces dates sont couvertes par ", fichier_entree
    )
  }
  indices
}

Ind_MiseEnRoute <- trouver_indices(date_debut_mise_en_route, as.character(as.Date(date_debut_calage) - 1), "mise en route")
Ind_Calage      <- trouver_indices(date_debut_calage, date_fin_calage, "calage")
Ind_Validation  <- trouver_indices(date_debut_validation, date_fin_validation, "validation")

FUN_CRIT <- switch(critere_calage,
  "NSE"     = ErrorCrit_NSE,
  "KGE"     = ErrorCrit_KGE,
  "KGE2012" = ErrorCrit_KGE2,
  stop("critere_calage doit valoir 'NSE', 'KGE' ou 'KGE2012' (valeur recue : ", critere_calage, ")")
)

RunOptions_Calage <- CreateRunOptions(
  FUN_MOD          = RunModel_GR4J,
  InputsModel      = InputsModel,
  IndPeriod_Run    = Ind_Calage,
  IndPeriod_WarmUp = Ind_MiseEnRoute,
  Outputs_Cal      = "Qsim"
)

InputsCrit <- CreateInputsCrit(
  FUN_CRIT    = FUN_CRIT,
  InputsModel = InputsModel,
  RunOptions  = RunOptions_Calage,
  Obs         = donnees$debit_mm[Ind_Calage]
)

CalibOptions <- CreateCalibOptions(FUN_MOD = RunModel_GR4J, FUN_CALIB = Calibration_Michel)

cat("Jours de mise en route :", length(Ind_MiseEnRoute),
    "| calage :", length(Ind_Calage),
    "| validation :", length(Ind_Validation), "\n")


### 4. Calage des parametres de GR4J

Le calage ajuste les 4 parametres du modele (X1 a X4) en optimisant le
critere choisi (`critere_calage`) sur la periode de calage.

In [ ]:
cat("\nCalage en cours (critere :", critere_calage, ", periode :",
    date_debut_calage, "->", date_fin_calage, ")...\n")

OutputsCalib <- Calibration_Michel(
  InputsModel  = InputsModel,
  RunOptions   = RunOptions_Calage,
  InputsCrit   = InputsCrit,
  CalibOptions = CalibOptions,
  FUN_MOD      = RunModel_GR4J
)

parametres_optimaux <- OutputsCalib$ParamFinalR
names(parametres_optimaux) <- c(
  "X1_capacite_production_mm",
  "X2_echange_souterrain_mm_j",
  "X3_capacite_routage_mm",
  "X4_temps_base_hydrogramme_j"
)

cat("\nParametres GR4J cales :\n")
print(round(parametres_optimaux, 2))
cat(
  "\n  X1 : capacite du reservoir de production -- plus grand = sol qui retient plus d'eau\n",
  "  X2 : echange en nappe (peut etre negatif = pertes, positif = apports exterieurs)\n",
  "  X3 : capacite du reservoir de routage -- regule le tarissement\n",
  "  X4 : temps de base de l'hydrogramme unitaire (j) -- inertie du bassin\n",
  sep = ""
)


### 5. Simulation et evaluation (calage et validation)

On rejoue le modele avec les parametres cales sur chacune des deux
periodes, et on calcule les indicateurs de performance usuels (NSE, KGE,
biais en %, RMSE) via `hydroGOF::gof`.

In [ ]:
simuler_et_evaluer <- function(indices_periode, nom_periode) {
  RunOptions <- CreateRunOptions(
    FUN_MOD          = RunModel_GR4J,
    InputsModel      = InputsModel,
    IndPeriod_Run    = indices_periode,
    IndPeriod_WarmUp = Ind_MiseEnRoute
  )
  OutputsModel <- RunModel_GR4J(
    InputsModel = InputsModel,
    RunOptions  = RunOptions,
    Param       = parametres_optimaux
  )

  obs <- donnees$debit_mm[indices_periode]
  sim <- OutputsModel$Qsim

  metriques <- suppressWarnings(hydroGOF::gof(sim, obs, method = "2012"))
  cat("\n--- Performance --", nom_periode, "---\n")
  print(round(metriques[c("NSE", "KGE", "PBIAS %", "RMSE"), , drop = FALSE], 3))

  list(
    outputs     = OutputsModel,
    dates       = donnees$date[indices_periode],
    obs         = obs,
    sim         = sim,
    metriques   = metriques,
    nom_periode = nom_periode
  )
}

resultats_calage     <- simuler_et_evaluer(Ind_Calage, "Calage")
resultats_validation <- simuler_et_evaluer(Ind_Validation, "Validation")


### 6. Graphiques de diagnostic

Chaque graphique est a la fois **affiche dans le notebook** et
**enregistre en PNG** dans `dossier_sortie` (pratique pour comparer entre
participant·e·s ou inclure dans un rapport).

In [ ]:
dir.create(dossier_sortie, showWarnings = FALSE, recursive = TRUE)

couleur_obs <- "#2a78d6"   # bleu
couleur_sim <- "#eb6834"   # orange


In [ ]:
# Diagnostic airGR standard (pluie, debits obs/sim, erreurs cumulees) -- Calage
png(file.path(dossier_sortie, paste0(nom_bassin, "_diagnostic_calage.png")), width = 1400, height = 1000, res = 130)
plot(resultats_calage$outputs, Qobs = resultats_calage$obs, main = paste(nom_bassin, "- Calage"))
dev.off()

# Apercu dans le notebook
plot(resultats_calage$outputs, Qobs = resultats_calage$obs, main = paste(nom_bassin, "- Calage"))


In [ ]:
# Diagnostic airGR standard -- Validation
png(file.path(dossier_sortie, paste0(nom_bassin, "_diagnostic_validation.png")), width = 1400, height = 1000, res = 130)
plot(resultats_validation$outputs, Qobs = resultats_validation$obs, main = paste(nom_bassin, "- Validation"))
dev.off()

# Apercu dans le notebook
plot(resultats_validation$outputs, Qobs = resultats_validation$obs, main = paste(nom_bassin, "- Validation"))


In [ ]:
# Hydrogramme observe vs simule, calage + validation sur un meme graphique
hydrogramme <- bind_rows(
  tibble(date = resultats_calage$dates,     periode = "Calage",     Observe = resultats_calage$obs,     Simule = resultats_calage$sim),
  tibble(date = resultats_validation$dates, periode = "Validation", Observe = resultats_validation$obs, Simule = resultats_validation$sim)
) %>%
  pivot_longer(cols = c(Observe, Simule), names_to = "serie", values_to = "debit_mm")

p_hydrogramme <- ggplot(hydrogramme, aes(x = date, y = debit_mm, color = serie)) +
  geom_line(linewidth = 0.4) +
  geom_vline(xintercept = as.Date(date_debut_validation), linetype = "dashed", color = "grey40") +
  scale_color_manual(values = c("Observe" = couleur_obs, "Simule" = couleur_sim)) +
  labs(
    title = paste("GR4J -", nom_bassin),
    subtitle = "Trait pointille = debut de la periode de validation",
    x = NULL, y = "Debit (mm/j)", color = NULL
  ) +
  theme_minimal(base_size = 12)

ggsave(file.path(dossier_sortie, paste0(nom_bassin, "_hydrogramme.png")), p_hydrogramme, width = 11, height = 5, dpi = 150)

p_hydrogramme


In [ ]:
# Nuage de points debit observe vs simule (calage et validation)
p_scatter <- bind_rows(
  tibble(periode = "Calage",     Observe = resultats_calage$obs,     Simule = resultats_calage$sim),
  tibble(periode = "Validation", Observe = resultats_validation$obs, Simule = resultats_validation$sim)
) %>%
  ggplot(aes(x = Observe, y = Simule)) +
  geom_point(alpha = 0.3, size = 0.8, color = couleur_obs) +
  geom_abline(slope = 1, intercept = 0, linetype = "dashed", color = "grey40") +
  facet_wrap(~periode) +
  coord_equal() +
  labs(
    title = paste("GR4J -", nom_bassin, "- debit observe vs simule"),
    x = "Debit observe (mm/j)", y = "Debit simule (mm/j)"
  ) +
  theme_minimal(base_size = 12)

ggsave(file.path(dossier_sortie, paste0(nom_bassin, "_obs_vs_sim.png")), p_scatter, width = 9, height = 5, dpi = 150)

p_scatter


### 7. Export des resultats

Parametres cales, table de performance (calage + validation) et series
simulees, ecrits en CSV dans `dossier_sortie`.

In [ ]:
write_csv(
  tibble(parametre = names(parametres_optimaux), valeur = as.numeric(parametres_optimaux)),
  file.path(dossier_sortie, paste0(nom_bassin, "_parametres_gr4j.csv"))
)

table_performance <- bind_rows(
  tibble(periode = "Calage",     as_tibble(t(resultats_calage$metriques[c("NSE", "KGE", "PBIAS %", "RMSE"), , drop = FALSE]))),
  tibble(periode = "Validation", as_tibble(t(resultats_validation$metriques[c("NSE", "KGE", "PBIAS %", "RMSE"), , drop = FALSE])))
)
write_csv(table_performance, file.path(dossier_sortie, paste0(nom_bassin, "_performance.csv")))
table_performance

write_csv(
  bind_rows(
    tibble(date = resultats_calage$dates,     periode = "Calage",     debit_observe_mm = resultats_calage$obs,     debit_simule_mm = resultats_calage$sim),
    tibble(date = resultats_validation$dates, periode = "Validation", debit_observe_mm = resultats_validation$obs, debit_simule_mm = resultats_validation$sim)
  ),
  file.path(dossier_sortie, paste0(nom_bassin, "_series_simulees.csv"))
)

cat("\nTermine. Resultats (CSV + PNG) ecrits dans :", normalizePath(dossier_sortie), "\n")


---

**Avant l'atelier** : executez ce notebook tel quel (parametres par
defaut = jeu de donnees exemple fictif) et verifiez qu'il se termine sans
erreur. Remplacez ensuite, dans la cellule « Parametres utilisateur »,
`fichier_entree`, `nom_bassin`, `superficie_km2` et les dates par vos
propres donnees (colonnes `date, pcp, evap, debit`).

**A venir** : un notebook de preparation des entrees GR4J a partir du
debit GloFAS (voir `download_glofas_data.ipynb`) et d'une reanalyse
pluie/ETP, pour enchainer directement extraction GloFAS -> modelisation
GR4J.